In [1]:
import pandas as pd
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from matplotlib import pyplot as plt

In [2]:
np.random.seed(42)

## Preprocessing for Logistic Regression

Preprocessing steps for the logistic regression model includes:

- Transformation: Since `title` and `content` are both text-based data, they must be transformed into vectors using TF-IDF. 
    - Normally, I would also be encoding `type` using One Hot Encoding which would ascribe each type category a number from 0 to 11. However, sklearn handles target feature transformation under the hood.



# Handling Missing Titles
The dataset has 35 rows with a missing title (roughly 0.12% of the total dataset). I considered using a `SimpleImputer` to handle this which would replace the NaN values with empty strings, but this introduced issues when used alongside `ColumnTransformer` due to `SimpleImputer`'s 2-D input requirement. 

Instead of dropping these 35 rows from the dataset I decided to manually fill them, as can be seen below.

In [ ]:
news = pd.read_csv("../datasets/audited_data.csv")
news["title"] = news["title"].fillna("")

In [ ]:
X = news[["title", "content"]]
y = news["type"]

# Preprocessing Pipeline

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2
)

print(X_train.shape)
print(y_train.shape)

(24000, 2)
(24000,)


In [6]:
preprocessor = ColumnTransformer([
    ("title", TfidfVectorizer(), "title"),
    ("content", TfidfVectorizer(), "content")
])

In [7]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter = 1000
    ))
])


In [10]:
stratified_kfold_cv = StratifiedKFold(
    n_splits = 10,
    shuffle = True,
    random_state = 42
)

In [11]:
import time

t0 = time.time()
pipeline.fit(X_train, y_train)
print(f"single fit: {time.time() - t0:1f}s")

single fit: 40.472805s


In [ ]:
metrics = {
    "f1_macro": "f1_macro",
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
}

In [16]:
results = cross_validate(
    pipeline,
    X_train,
    y_train,
    cv = stratified_kfold_cv,
    scoring = metrics
    )


for m in metrics:
    s = results[f"test_{m}"]
    print(f"{m:16s} {s.mean():.4f} ± {s.std():.4f}")

f1_macro         0.8127 ± 0.0076
accuracy         0.8135 ± 0.0074
precision_macro  0.8179 ± 0.0078
